<a href="https://colab.research.google.com/github/ajaykumar080286/DeepLearning/blob/master/34_cat_dog_normal_augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential

from keras.layers import Dense, Dropout,Flatten,MaxPool2D, Conv2D,BatchNormalization, Activation, MaxPool2D # Added Activation
import matplotlib.pyplot as plt
import seaborn as sns


import zipfile
import requests
import os

In [11]:
# Path to your zip file (URL)
zip_url = "https://raw.githubusercontent.com/ajaykumar080286/DeepLearning/master/dogs_vs_cats_Augmentation.zip"

# Local path to save the downloaded zip file
local_zip_path = "/content/dogs_vs_cats.zip"

# Download the zip file
print(f"Downloading {zip_url} to {local_zip_path}...")
response = requests.get(zip_url)
response.raise_for_status() # Raise an exception for HTTP errors
with open(local_zip_path, 'wb') as f:
    f.write(response.content)
print("Download complete.")

# Open the zip file
zip_ref = zipfile.ZipFile(local_zip_path, 'r')
zip_ref.extractall("/content")
zip_ref.close()
print("Extraction complete.")

# Optionally, remove the downloaded zip file after extraction
os.remove(local_zip_path)

Download complete.
Extraction complete.


In [12]:
mkdir = "/content/dogs_vs_cats/train"
category=['cats','dogs']

In [27]:
import os
import cv2

data = []
base_path = "/content/dogs_vs_cats/train"   # your dataset root
categories = ["cats", "dogs"]               # folder names

for i, category in enumerate(categories):
    folder_path = os.path.join(base_path, category)
    print(folder_path)
    label = i   # 0 for cats, 1 for dogs

    for j in os.listdir(folder_path):
        img_path = os.path.join(folder_path, j)
        print(img_path)

        img = cv2.imread(img_path)


        if img is None:   # skip unreadable files
            continue
        img = cv2.resize(img, (150, 150))  # force same size
        data.append([img, label])

print(f"Loaded {len(data)} images")

/content/dogs_vs_cats/train/cats
/content/dogs_vs_cats/train/cats/cat.11.jpg
/content/dogs_vs_cats/train/dogs
/content/dogs_vs_cats/train/dogs/dog.12485.jpg
Loaded 2 images


In [28]:
IMG_SIZE = 256
X = []
y = []

for i in data:
    X.append(i[0])
    y.append(i[1])

X = np.array(X, dtype=np.float32) / 255.0   # normalize
y = np.array(y)

In [29]:
X.shape

(2, 150, 150, 3)

In [30]:
y.shape

(2,)

In [31]:
model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape=(150, 150, 3)))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(2, 2)))

model.add(Conv2D(32, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(2, 2)))

model.add(Conv2D(64, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(2, 2)))


model.add(Flatten())  # this converts our 3D feature maps to 1D feature vectors
model.add(Dense(64))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(1))
model.add(Activation('sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [32]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_10 (Activation)      │ (None, 148, 148, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 72, 72, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_11 (Activation)      │ (None, 72, 72, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 34, 34, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_12 (Activation)      │ (None, 34, 34, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │     1,183,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_13 (Activation)      │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_14 (Activation)      │ (None, 1)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,212,513 (4.63 MB)

 Trainable params: 1,212,513 (4.63 MB)

 Non-trainable params: 0 (0.00 B)

In [33]:
model.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [34]:
model.fit(X,y,epochs=5, validation_split=0.1)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.6647 - val_accuracy: 0.0000e+00 - val_loss: 3.6025
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 1.0000 - loss: 0.0649 - val_accuracy: 0.0000e+00 - val_loss: 11.6343
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - accuracy: 1.0000 - loss: 9.7578e-08 - val_accuracy: 0.0000e+00 - val_loss: 11.6343
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 1.0000 - loss: 3.9486e-05 - val_accuracy: 0.0000e+00 - val_loss: 11.6748
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 1.0000 - loss: 9.6634e-08 - val_accuracy: 0.0000e+00 - val_loss: 11.6749


**With Data Augmentation**

In [35]:
from keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [36]:
batch_size=16


train_dataset = ImageDataGenerator(
        rescale=1./255,
        rotation_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True
        )


In [37]:
test_dataset = ImageDataGenerator(rescale=1./255)

In [39]:
train_generator = train_dataset.flow_from_directory(
        '/content/dogs_vs_cats/train',  # this is the target directory
        target_size=(150, 150),  # all images will be resized to 150x150
        batch_size=batch_size,
        class_mode='binary')  # sin
validation_generator = test_dataset.flow_from_directory(
        '/content/dogs_vs_cats/test',
        target_size=(150, 150),
        batch_size=batch_size,
        class_mode='binary')


Found 2 images belonging to 2 classes.
Found 12 images belonging to 2 classes.


In [40]:
model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape=(150, 150, 3)))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(2, 2)))

model.add(Conv2D(32, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(2, 2)))

model.add(Conv2D(64, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Flatten())  # this converts our 3D feature maps to 1D feature vectors
model.add(Dense(64))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(1))
model.add(Activation('sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [41]:
model.fit(
        train_generator,
        steps_per_epoch=2000 // batch_size,
        epochs=25,
        validation_data=validation_generator,
        validation_steps=800 // batch_size)


Epoch 1/25
  1/125 ━━━━━━━━━━━━━━━━━━━━ 4:07 2s/step - accuracy: 1.0000 - loss: 0.6512

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 1.0000 - loss: 0.6512 - val_accuracy: 0.9167 - val_loss: 0.3090
Epoch 2/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5000 - loss: 2.0876 - val_accuracy: 0.0833 - val_loss: 2.7047
Epoch 3/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5000 - loss: 2.0375 - val_accuracy: 0.9167 - val_loss: 0.6568
Epoch 4/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5000 - loss: 0.6795 - val_accuracy: 0.0833 - val_loss: 0.9612
Epoch 5/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5000 - loss: 0.6264 - val_accuracy: 0.9167 - val_loss: 0.4212
Epoch 6/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5000 - loss: 0.4685 - val_accuracy: 0.9167 - val_loss: 0.5491
Epoch 7/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 1.0000 - loss: 0.3839 - val_accuracy: 0.0833 - val_loss: 1.0109
Epoch 8/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5000 - loss: 0.6526 - val_accuracy: 0.0833 - val_

In [42]:
#https://blog.keras.io/building-powerful-image-classification-models-using-very-little-data.html